In [ ]:
from datasets import load_dataset
import json

# Save Data to Jsonl

In [2]:
en_dataset = load_dataset("omarkamali/wikipedia-monthly", "latest.en", split="train", streaming=True)

Resolving data files:   0%|          | 0/1425 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1432 [00:00<?, ?it/s]

In [4]:
shuffled = en_dataset.shuffle(buffer_size=10_000, seed=42)

In [ ]:

max_articles_to_process = 10_000  # Start small to test your pipeline
output_file = "rag_corpus.jsonl"

print(f"Starting extraction to {output_file}...")

with open(output_file, "w", encoding="utf-8") as f:
    for i, article in enumerate(shuffled):
        # Stop condition for testing
        if i >= max_articles_to_process:
            break
            
        # Extract relevant fields (adjust based on the dataset's specific schema)
        title = article.get('title', 'Unknown Title')
        text = article.get('text', '')
        url = article.get('url', '')
        id = article.get('id',  i)

        if not text.strip():
            continue

        # 5. Format and save each chunk with essential metadata
        record = {
            "id": id,
            "title": title,
            "url": url,
            "text": text,
        }
        # Write as JSON Lines (JSONL) - ideal for large, iterative datasets
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Successfully processed {max_articles_to_process} articles into chunks.")

Starting extraction to rag_corpus.jsonl...
Successfully processed 10000 articles into chunks.


In [7]:
id_dataset = load_dataset("omarkamali/wikipedia-monthly", "latest.id", split="train", streaming=True)
id_shuffled = id_dataset.shuffle(buffer_size=10_000, seed=42)

Resolving data files:   0%|          | 0/1425 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/153 [00:00<?, ?it/s]

In [9]:

max_articles_to_process = 100_000  # Start small to test your pipeline
output_file = "id_rag_corpus.jsonl"

print(f"Starting extraction to {output_file}...")

with open(output_file, "w", encoding="utf-8") as f:
    for i, article in enumerate(id_shuffled):
        # Stop condition for testing
        if i >= max_articles_to_process:
            break
            
        # Extract relevant fields (adjust based on the dataset's specific schema)
        title = article.get('title', 'Unknown Title')
        text = article.get('text', '')
        url = article.get('url', '')
        id = article.get('id',  i)

        if not text.strip():
            continue

        # 5. Format and save each chunk with essential metadata
        record = {
            "id": id,
            "title": title,
            "url": url,
            "text": text,
        }
        # Write as JSON Lines (JSONL) - ideal for large, iterative datasets
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Successfully processed {max_articles_to_process} articles into chunks.")

Starting extraction to id_rag_corpus.jsonl...
Successfully processed 100000 articles into chunks.


# Insert data to ChromaDB

In [5]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import json

In [2]:
client = chromadb.PersistentClient()
collection = client.get_or_create_collection(name='file_corpus')

In [9]:
def recursive_chunk(text: str, chunk_size: int, overlap: int) -> List[str]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        is_separator_regex=False,
    )
    documents = text_splitter.create_documents([text])
    chunked_texts = [doc.page_content for doc in documents]
    return chunked_texts 

def ingest_jsonl_in_batches(filepaths, chroma_collection, batch_size):
    for file_path in filepaths:
        if not os.path.exists(file_path):
            print(f"Warning: File not found -> {file_path}")
            continue

        print(f"--- Processing {file_path} ---")
        
        # Renamed accumulators to avoid confusion
        batch_documents = []
        batch_metadatas = []
        batch_ids = []
        
        inserted_count = 0
        line_count = 0

        # Open the file and process it line-by-line (streaming)
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue # Skip empty lines
                
                line_count += 1
                
                # Parse the individual line
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"  -> Skipping line {line_count}: Invalid JSON format.")
                    continue

                # --- Adjust keys to match your JSONL schema ---
                text_content = item.get("text", "") 
                if not text_content:
                    continue 
                
                # Extract metadata, ensuring values are primitive types
                metadata = {k: v for k, v in item.items() if k != "text" and isinstance(v, (str, int, float, bool))}
                base_doc_id = str(item.get("id", f"doc_{line_count}"))

                # 1. Generate chunks
                chunks = recursive_chunk(text_content, 512, int(0.2 * 512))
                
                # 2. Iterate over the resulting chunks
                for i, chunk in enumerate(chunks):
                    # Extract the string from the Langchain Document object
                    batch_documents.append(chunk) 
                    
                    # Duplicate the metadata for each chunk
                    batch_metadatas.append(metadata)
                    
                    # Append a unique ID for each chunk (e.g., doc_1_chunk_0)
                    batch_ids.append(f"{base_doc_id}_chunk_{i}")

                    # 3. Insert batch when the limit is reached
                    # Moved INSIDE the chunk loop so we don't accidentally exceed batch_size
                    if len(batch_documents) >= batch_size:
                        chroma_collection.add(
                            documents=batch_documents,
                            metadatas=batch_metadatas,
                            ids=batch_ids
                        )
                        inserted_count += len(batch_documents)
                        print(f"  -> Inserted {inserted_count} records so far...")
                        
                        batch_documents.clear()
                        batch_metadatas.clear()
                        batch_ids.clear()

        # Catch and insert the final remainder batch after the file ends
        if batch_documents:
            chroma_collection.add(
                documents=batch_documents,
                metadatas=batch_metadatas,
                ids=batch_ids
            )
            inserted_count += len(batch_documents)
            print(f"  -> Inserted final batch. Total: {inserted_count} records.")

        print(f"Successfully finished {file_path}.\n")

In [ ]:
# Run the ingestion
corpus = ["rag_corpus.jsonl", "id_rag_corpus.jsonl"]
ingest_jsonl_in_batches(corpus, collection, batch_size=1000)

print("All files processed. Database is ready for querying.")

--- Processing rag_corpus.jsonl ---
  -> Inserted 1000 records so far...
  -> Inserted 2000 records so far...
  -> Inserted 3000 records so far...
  -> Inserted 4000 records so far...
  -> Inserted 5000 records so far...
  -> Inserted 6000 records so far...
  -> Inserted 7000 records so far...
  -> Inserted 8000 records so far...
  -> Inserted 9000 records so far...
  -> Inserted 10000 records so far...
  -> Inserted 11000 records so far...
  -> Inserted 12000 records so far...
  -> Inserted 13000 records so far...
  -> Inserted 14000 records so far...
  -> Inserted 15000 records so far...
  -> Inserted 16000 records so far...
  -> Inserted 17000 records so far...
  -> Inserted 18000 records so far...
  -> Inserted 19000 records so far...
  -> Inserted 20000 records so far...
  -> Inserted 21000 records so far...
  -> Inserted 22000 records so far...
  -> Inserted 23000 records so far...
  -> Inserted 24000 records so far...
  -> Inserted 25000 records so far...
  -> Inserted 26000 rec

In [1]:
import json

def read_corpus_samples(file_path, num_samples=50):
    """
    Read and display JSON samples from a JSONL file.
    
    Args:
        file_path (str): Path to the JSONL file
        num_samples (int): Number of samples to read (default: 50)
    """
    samples = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= num_samples:
                    break
                try:
                    sample = json.loads(line.strip())
                    samples.append(sample)
                except json.JSONDecodeError as e:
                    print(f"Error parsing line {i+1}: {e}")
        
        # Display the samples
        print(f"Successfully read {len(samples)} samples from {file_path}\n")
        
        for idx, sample in enumerate(samples, 1):
            print(f"\n{'='*80}")
            print(f"Sample {idx}:")
            print(f"{'='*80}")
            print(json.dumps(sample, ensure_ascii=False, indent=2))
        
        return samples
    
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []




In [2]:
corpus_path = r"corpus\id_rag_corpus.jsonl"
samples = read_corpus_samples(corpus_path, num_samples=50)

print(f"\n\nTotal samples read: {len(samples)}")

Successfully read 50 samples from corpus\id_rag_corpus.jsonl


Sample 1:
{
  "id": "1572328",
  "title": "Daftar negara menurut jumlah pengguna Internet",
  "url": "https://id.wikipedia.org/wiki/Daftar_negara_menurut_jumlah_pengguna_Internet",
  "text": "Berikut adalah daftar negara menurut jumlah pengguna Internet pada tahun 2021.\n\nPengguna Internet adalah orang yang menggunakan Internet dalam 12 bulan terakhir melalui perangkat apapun, termasuk telepon genggam. Penetrasi adalah persentase penduduk negara yang menggunakan Internet. Perkiraan diambil dari survei rumah-ke-rumah atau data langganan Internet.\n\nNon-negara dan wilayah sengketa dimiringkan. Semua negara anggota Perserikatan Bangsa-Bangsa disertakan, kecuali Sudan Selatan. Taiwan dicantumkan sebagai negara berdaulat.\n\nDaftar\nNo.Negara atau wilayahJumlahPopulasiPersentase pengguna Internet NoPer.%Per.11,120,000,0001,411,750,000179.33%982718,740,0001,366,417,754254.29%1203323,560,000331,893,745397.48%704274,200,000280,72